# 🧹 Day 02 — Data Cleaning & Preprocessing
## 30 Days of Machine Learning

**Dataset:** Messy Employee / HR Data (500 rows, 12 columns)  
**Goal:** Take a real-world messy dataset and clean it from scratch — step by step.

**What we'll do:**
- ✅ Explore & understand the mess
- ✅ Remove duplicates
- ✅ String cleaning (names, gender, department, city)
- ✅ Fix data types & inconsistent encoding
- ✅ Handle missing values (fill, drop, impute)
- ✅ Detect & remove outliers (IQR + Z-score)
- ✅ Before vs After comparison

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

COLORS = {
    'dirty':  '#E74C3C',
    'clean':  '#2ECC71',
    'neutral':'#3498DB',
    'warn':   '#F39C12'
}

print('✅ Libraries loaded!')

---
## 📂 Step 2 — Load & First Look at the Messy Data

In [ ]:
df_raw = pd.read_csv('messy_hr_data.csv')

print(f'📊 Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'\n📋 Columns: {df_raw.columns.tolist()}')
df_raw.head(10)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe().round(2)

In [ ]:
# ── Full audit of all problems ──
print('=' * 55)
print('        🔍 DATA QUALITY AUDIT REPORT')
print('=' * 55)

# Missing values
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().mean() * 100).round(2)
print('\n📌 Missing Values:')
for col in df_raw.columns:
    if missing[col] > 0:
        print(f'   {col:<22} {missing[col]:>4} nulls  ({missing_pct[col]}%)')

# Duplicates
dups = df_raw.duplicated(subset='employee_id').sum()
print(f'\n📌 Duplicate employee_ids : {dups}')
print(f'📌 Fully duplicate rows   : {df_raw.duplicated().sum()}')

# Inconsistent categories
print(f'\n📌 Unique values in "gender"     : {sorted(df_raw["gender"].dropna().unique())}')
print(f'📌 Unique values in "department" : {sorted(df_raw["department"].dropna().unique())}')
print(f'📌 Unique values in "city"       : {sorted(df_raw["city"].dropna().unique())}')
print(f'📌 Unique values in "education"  : {sorted(df_raw["education"].dropna().unique())}')

print('=' * 55)

In [ ]:
# ── Visualize missing values ──
fig, ax = plt.subplots(figsize=(10, 5))

miss_df = pd.DataFrame({'col': missing.index, 'count': missing.values, 'pct': missing_pct.values})
miss_df = miss_df[miss_df['count'] > 0].sort_values('pct', ascending=True)

bars = ax.barh(miss_df['col'], miss_df['pct'],
               color=[COLORS['dirty'] if p > 10 else COLORS['warn'] for p in miss_df['pct']],
               edgecolor='white', height=0.6)

for bar, val, cnt in zip(bars, miss_df['pct'], miss_df['count']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}%  ({cnt} rows)', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Missing %', fontsize=12)
ax.set_title('🔍 Missing Values by Column — BEFORE Cleaning', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, 20)
plt.tight_layout()
plt.savefig('missing_before.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: missing_before.png')

---
## 🧹 Step 3 — Remove Duplicates

In [ ]:
df = df_raw.copy()

before = len(df)

# Drop fully duplicate rows first
full_dups = df.duplicated().sum()
df.drop_duplicates(inplace=True)

# Keep first occurrence of duplicate employee_ids
id_dups = df.duplicated(subset='employee_id').sum()
df.drop_duplicates(subset='employee_id', keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)

after = len(df)
print(f'✅ Full duplicate rows removed  : {full_dups}')
print(f'✅ Duplicate employee_ids removed: {id_dups}')
print(f'✅ Rows: {before} → {after}  (removed {before - after})')

---
## ✏️ Step 4 — String Cleaning & Standardization

In [ ]:
# ── Clean Name column ──
df['name'] = df['name'].str.strip().str.title()
print('✅ Names cleaned (stripped whitespace, Title Case)')
print(df['name'].head(5).tolist())

In [ ]:
# ── Standardize Gender ──
print('Before:', df['gender'].value_counts().to_dict())

gender_map = {
    'male': 'Male', 'm': 'Male', 'MALE': 'Male',
    'female': 'Female', 'f': 'Female', 'FEMALE': 'Female', 'Female': 'Female', 'Male': 'Male'
}
df['gender'] = df['gender'].str.strip().map(gender_map)

print('After: ', df['gender'].value_counts().to_dict())
print('✅ Gender standardized → Male / Female')

In [ ]:
# ── Standardize Department ──
print('Before:', sorted(df['department'].str.strip().str.lower().unique()))

dept_map = {
    'engineering': 'Engineering', 'eng': 'Engineering',
    'marketing': 'Marketing', 'mktg': 'Marketing',
    'hr': 'HR', 'human resources': 'HR',
    'sales': 'Sales',
    'finance': 'Finance', 'fin': 'Finance',
    'operations': 'Operations'
}
df['department'] = df['department'].str.strip().str.lower().map(dept_map)

print('After: ', sorted(df['department'].dropna().unique()))
print('✅ Department standardized → 6 clean categories')

In [ ]:
# ── Standardize City ──
print('Before:', sorted(df['city'].str.strip().unique()))

city_map = {
    'new york': 'New York', 'nyc': 'New York',
    'san francisco': 'San Francisco', 'sf': 'San Francisco',
    'chicago': 'Chicago',
    'austin': 'Austin',
    'boston': 'Boston'
}
df['city'] = df['city'].str.strip().str.lower().map(city_map)

print('After: ', sorted(df['city'].dropna().unique()))
print('✅ City standardized → 5 clean categories')

In [ ]:
# ── Standardize Education ──
print('Before:', sorted(df['education'].str.strip().unique()))

edu_map = {
    'bachelor': 'Bachelor', 'bachelors': 'Bachelor', 'b.tech': 'Bachelor',
    'master': 'Master', 'masters': 'Master', 'mba': 'Master',
    'phd': 'PhD',
    'high school': 'High School', 'hs': 'High School'
}
df['education'] = df['education'].str.strip().str.lower().map(edu_map)

print('After: ', sorted(df['education'].dropna().unique()))
print('✅ Education standardized → 4 clean categories')

In [ ]:
# ── Standardize Phone ──
def clean_phone(phone):
    if pd.isna(phone): return np.nan
    digits = ''.join(filter(str.isdigit, str(phone)))
    if len(digits) == 10:
        return f'({digits[:3]}) {digits[3:6]}-{digits[6:]}'
    return np.nan  # invalid

df['phone'] = df['phone'].apply(clean_phone)
print(f'✅ Phone numbers standardized to (XXX) XXX-XXXX format')
print(df['phone'].dropna().head(5).tolist())

---
## 📅 Step 5 — Fix Data Types (Join Date)

In [ ]:
print('Sample raw join_dates:', df['join_date'].sample(10).tolist())

# Replace invalid strings with NaN
invalid_dates = ['TBD', 'N/A', '', 'unknown', '99/99/9999']
df['join_date'] = df['join_date'].replace(invalid_dates, np.nan)

# Parse multiple formats
df['join_date'] = pd.to_datetime(df['join_date'], format='mixed', errors='coerce')

# Extract useful features
df['join_year']  = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month
df['tenure_years'] = (pd.Timestamp('2026-01-01') - df['join_date']).dt.days / 365.25
df['tenure_years'] = df['tenure_years'].round(2)

valid = df['join_date'].notna().sum()
print(f'\n✅ Dates parsed: {valid}/{len(df)} valid')
print(f'✅ New features: join_year, join_month, tenure_years')
print(df[['join_date','join_year','join_month','tenure_years']].dropna().head(5))

---
## 🔧 Step 6 — Handle Missing Values

In [ ]:
print('Missing values BEFORE imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Age → median imputation
age_median = df['age'].median()
df['age'].fillna(age_median, inplace=True)
print(f'\n✅ age: filled with median ({age_median:.0f})')

# Salary → median by department
df['salary'] = df.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.median())
)
# If still NaN (no dept), fill with global median
df['salary'].fillna(df['salary'].median(), inplace=True)
print('✅ salary: filled with department median')

# Years experience → median by department
df['years_experience'] = df.groupby('department')['years_experience'].transform(
    lambda x: x.fillna(x.median())
)
df['years_experience'].fillna(df['years_experience'].median(), inplace=True)
print('✅ years_experience: filled with department median')

# Performance score → median
df['performance_score'].fillna(df['performance_score'].median(), inplace=True)
print('✅ performance_score: filled with median')

# Gender → mode
df['gender'].fillna(df['gender'].mode()[0], inplace=True)
print('✅ gender: filled with mode')

# Department → mode
df['department'].fillna(df['department'].mode()[0], inplace=True)
df['city'].fillna(df['city'].mode()[0], inplace=True)
df['education'].fillna(df['education'].mode()[0], inplace=True)
print('✅ department / city / education: filled with mode')

print(f'\n🔍 Remaining nulls: {df.isnull().sum().sum()} (phone & dates are OK to keep)')

---
## 📊 Step 7 — Outlier Detection & Removal (IQR + Z-Score)

In [ ]:
# ── Visualize outliers BEFORE ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, color in zip(axes, ['age', 'salary', 'years_experience'],
                           [COLORS['dirty'], COLORS['warn'], COLORS['neutral']]):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor=COLORS['dirty'],
                               markersize=6, alpha=0.7))
    ax.set_title(f'{col}\n(BEFORE outlier removal)', fontsize=12, fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle('🚨 Outliers BEFORE Cleaning', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outliers_before.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outliers_before.png')

In [ ]:
from scipy import stats

def remove_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df)
    df_clean = df[(df[col] >= lower) & (df[col] <= upper)]
    removed = before - len(df_clean)
    print(f'  IQR  [{col}]: range [{lower:.1f}, {upper:.1f}] → removed {removed} outliers')
    return df_clean

def remove_outliers_zscore(df, col, threshold=3):
    z = np.abs(stats.zscore(df[col].dropna()))
    valid_idx = df[col].dropna().index[z < threshold]
    before = len(df)
    df_clean = df[df.index.isin(valid_idx) | df[col].isna()]
    removed = before - len(df_clean)
    print(f'  Z-score [{col}]: threshold={threshold} → removed {removed} outliers')
    return df_clean

before_total = len(df)
print('🔧 Removing outliers...')

# Age: IQR (biological limits)
df = remove_outliers_iqr(df, 'age')

# Salary: Z-score (financial outliers)
df = remove_outliers_zscore(df, 'salary', threshold=3)

# Years experience: IQR
df = remove_outliers_iqr(df, 'years_experience')

df.reset_index(drop=True, inplace=True)
print(f'\n✅ Total rows after outlier removal: {before_total} → {len(df)}')

In [ ]:
# ── Visualize AFTER outlier removal ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, color in zip(axes, ['age', 'salary', 'years_experience'],
                           [COLORS['clean'], COLORS['clean'], COLORS['clean']]):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor=COLORS['warn'],
                               markersize=6, alpha=0.7))
    ax.set_title(f'{col}\n(AFTER outlier removal)', fontsize=12, fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle('✅ Outliers AFTER Cleaning', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outliers_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outliers_after.png')

---
## 🔢 Step 8 — Encode Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

# Binary encoding for gender
df_encoded['gender_enc'] = (df_encoded['gender'] == 'Male').astype(int)
print('✅ gender_enc: Male=1, Female=0')

# Label encoding for ordinals
edu_order = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
df_encoded['education_enc'] = df_encoded['education'].map(edu_order)
print('✅ education_enc: High School=0, Bachelor=1, Master=2, PhD=3')

perf_order = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
df_encoded['performance_enc'] = df_encoded['performance_score'].map(perf_order)

# One-hot encoding for department & city
dept_dummies = pd.get_dummies(df_encoded['department'], prefix='dept')
city_dummies = pd.get_dummies(df_encoded['city'], prefix='city')
df_encoded = pd.concat([df_encoded, dept_dummies, city_dummies], axis=1)

print(f'✅ One-hot encoded: department ({len(dept_dummies.columns)} cols), city ({len(city_dummies.columns)} cols)')
print(f'\n📊 Final encoded shape: {df_encoded.shape}')
print('\nNew columns added:')
new_cols = ['gender_enc', 'education_enc', 'performance_enc'] + list(dept_dummies.columns) + list(city_dummies.columns)
print(new_cols)

---
## 📈 Step 9 — Before vs After Visualizations

In [ ]:
# ── Plot: Data Quality Summary ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before vs After: row count
categories = ['Total Rows', 'Missing Values', 'Duplicates', 'Outliers']
before_vals = [len(df_raw), df_raw.isnull().sum().sum(),
               df_raw.duplicated(subset='employee_id').sum(), 30]
after_vals  = [len(df), df[['age','salary','years_experience','gender','department']].isnull().sum().sum(),
               0, 0]

x = np.arange(len(categories))
w = 0.35
bars1 = axes[0].bar(x - w/2, before_vals, w, label='Before', color=COLORS['dirty'], edgecolor='white', alpha=0.85)
bars2 = axes[0].bar(x + w/2, after_vals,  w, label='After',  color=COLORS['clean'], edgecolor='white', alpha=0.85)

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold', color=COLORS['dirty'])
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold', color=COLORS['clean'])

axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, fontsize=11)
axes[0].set_title('Data Quality: Before vs After', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# Clean department distribution
dept_counts = df['department'].value_counts()
bars = axes[1].bar(dept_counts.index, dept_counts.values,
                   color=sns.color_palette('Set2', len(dept_counts)),
                   edgecolor='white', width=0.6)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Employee Count by Department (Clean)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Department', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('cleaning_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: cleaning_summary.png')

In [ ]:
# ── Salary distribution: Before vs After ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw['salary'].dropna(), bins=40, color=COLORS['dirty'], edgecolor='white', alpha=0.8)
axes[0].set_title('Salary Distribution — BEFORE', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Salary ($)'); axes[0].set_ylabel('Count')
axes[0].axvline(df_raw['salary'].median(), color='black', lw=2, linestyle='--',
                label=f'Median: ${df_raw["salary"].median():,.0f}')
axes[0].legend()

axes[1].hist(df['salary'].dropna(), bins=40, color=COLORS['clean'], edgecolor='white', alpha=0.8)
axes[1].set_title('Salary Distribution — AFTER', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Salary ($)'); axes[1].set_ylabel('Count')
axes[1].axvline(df['salary'].median(), color='black', lw=2, linestyle='--',
                label=f'Median: ${df["salary"].median():,.0f}')
axes[1].legend()

plt.suptitle('💰 Salary: Before vs After Cleaning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('salary_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: salary_before_after.png')

In [ ]:
# ── Clean data insights ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Salary by department
dept_salary = df.groupby('department')['salary'].median().sort_values(ascending=True)
axes[0].barh(dept_salary.index, dept_salary.values,
             color=sns.color_palette('Set2', len(dept_salary)), edgecolor='white', height=0.6)
axes[0].set_title('Median Salary by Department', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Salary ($)')
for i, v in enumerate(dept_salary.values):
    axes[0].text(v + 500, i, f'${v:,.0f}', va='center', fontsize=9, fontweight='bold')

# Age distribution clean
axes[1].hist(df['age'], bins=25, color=COLORS['neutral'], edgecolor='white', alpha=0.85)
axes[1].axvline(df['age'].mean(), color=COLORS['dirty'], lw=2, linestyle='--',
                label=f'Mean: {df["age"].mean():.1f}')
axes[1].set_title('Age Distribution (Clean)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Age'); axes[1].legend()

# Education distribution
edu_order_list = ['High School', 'Bachelor', 'Master', 'PhD']
edu_counts = df['education'].value_counts().reindex(edu_order_list).dropna()
axes[2].bar(edu_counts.index, edu_counts.values,
            color=['#E74C3C', '#F39C12', '#3498DB', '#9B59B6'],
            edgecolor='white', width=0.6)
axes[2].set_title('Education Level Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Education')
for i, v in enumerate(edu_counts.values):
    axes[2].text(i, v + 1, str(v), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('📊 Clean HR Data Insights', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('clean_insights.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: clean_insights.png')

---
## 💾 Step 10 — Save Clean Dataset

In [ ]:
# Save the clean version
df.to_csv('clean_hr_data.csv', index=False)
df_encoded.to_csv('encoded_hr_data.csv', index=False)

print('✅ Saved: clean_hr_data.csv')
print('✅ Saved: encoded_hr_data.csv')

# Final summary report
print('\n' + '=' * 55)
print('        ✅ FINAL DATA CLEANING REPORT')
print('=' * 55)
print(f'  Original rows          : {len(df_raw)}')
print(f'  Clean rows             : {len(df)}')
print(f'  Rows removed           : {len(df_raw) - len(df)}')
print(f'  Columns (raw)          : {df_raw.shape[1]}')
print(f'  Columns (clean)        : {df.shape[1]}')
print(f'  Columns (encoded)      : {df_encoded.shape[1]}')
print(f'  Missing values removed : {df_raw.isnull().sum().sum()}')
print(f'  Duplicates removed     : {df_raw.duplicated(subset="employee_id").sum()}')
print(f'  Categorical columns    : 5 standardized')
print(f'  Date columns parsed    : 1 (join_date)')
print(f'  New features created   : tenure_years, join_year, join_month')
print('=' * 55)
print('\n📋 Clean Data Sample:')
df[['name','age','gender','department','salary','education','performance_score']].head(8)

---
## 📝 Summary — What We Learned

| Problem Found | Solution Applied |
|---|---|
| **Duplicate employee IDs** | `drop_duplicates(subset='employee_id')` |
| **Inconsistent gender** (M/F/male/MALE) | `.str.strip().map(gender_map)` |
| **Messy departments** (eng/Eng/engineering) | `.str.lower().map(dept_map)` |
| **Multiple date formats** | `pd.to_datetime(infer_datetime_format=True)` |
| **Missing numeric values** | Median imputation (by department group) |
| **Missing categoricals** | Mode imputation |
| **Salary outliers** (₹999,999) | Z-score method (threshold=3) |
| **Age outliers** (age=999) | IQR method (1.5×IQR fence) |
| **Categorical encoding** | One-hot (dept/city) + ordinal (education) |

---

## 📁 Files to Upload to GitHub

```
day02-data-cleaning/
├── hr_data_cleaning.ipynb   ← this notebook
├── messy_hr_data.csv        ← raw messy input
├── clean_hr_data.csv        ← cleaned output
├── encoded_hr_data.csv      ← ML-ready output
├── missing_before.png
├── outliers_before.png
├── outliers_after.png
├── cleaning_summary.png
├── salary_before_after.png
├── clean_insights.png
└── README.md
```

---
*Day 2/30 — #30DaysOfML 🚀*